# Notebook 03 — DML nas tabelas Delta (INSERT / UPDATE / DELETE)

Demonstra as três operações de DML nas tabelas Delta do bucket `bronze`, além de **Time Travel** e **DESCRIBE HISTORY** para auditoria.

**Tabelas utilizadas:** `cliente`, `apolice`, `sinistro`

## 1. Imports e SparkSession

In [ ]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_date
from delta import configure_spark_with_delta_pip, DeltaTable

load_dotenv()

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT', 'http://localhost:9020')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY', 'minioadmin')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY', 'minioadmin')
BRONZE_BUCKET    = os.getenv('MINIO_BRONZE_BUCKET', 'bronze')

S3A_ENDPOINT = MINIO_ENDPOINT.replace('http://', '').replace('https://', '')

builder = (
    SparkSession.builder
    .appName('dml-delta-bronze')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.hadoop.fs.s3a.endpoint', f'http://{S3A_ENDPOINT}')
    .config('spark.hadoop.fs.s3a.access.key', MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key', MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .config('spark.jars.packages',
            'io.delta:delta-spark_2.12:3.2.0,'
            'org.apache.hadoop:hadoop-aws:3.3.4,'
            'com.amazonaws:aws-java-sdk-bundle:1.12.262')
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('WARN')

def bronze_path(tabela):
    return f's3a://{BRONZE_BUCKET}/{tabela}'

print(f'Spark {spark.version} iniciado!')

---
## 2. INSERT — Novos clientes e apólices

Adicionamos 3 novos clientes e 2 novas apólices ao bronze.

In [ ]:
# Estado anterior
dt_cliente = DeltaTable.forPath(spark, bronze_path('cliente'))
antes = spark.read.format('delta').load(bronze_path('cliente')).count()
print(f'Clientes antes do INSERT: {antes}')

# Novos registros
novos_clientes = spark.createDataFrame([
    (61, 'Patricia Oliveira',  '611.611.611-61', '1992-04-15', 'patricia.oliveira@email.com',  'F'),
    (62, 'Rodrigo Almeida',   '622.622.622-62', '1985-09-28', 'rodrigo.almeida@email.com',    'M'),
    (63, 'Samantha Ferreira', '633.633.633-63', '1998-01-07', 'samantha.ferreira@email.com',  'F'),
], ['id', 'nome', 'cpf', 'data_nascimento', 'email', 'sexo'])

novos_clientes.write.format('delta').mode('append').save(bronze_path('cliente'))

depois = spark.read.format('delta').load(bronze_path('cliente')).count()
print(f'Clientes depois do INSERT: {depois} (+{depois - antes})')

In [ ]:
# INSERT em apolice
antes_ap = spark.read.format('delta').load(bronze_path('apolice')).count()
print(f'Apólices antes do INSERT: {antes_ap}')

novas_apolices = spark.createDataFrame([
    (31, 61, 1,  'APL-2024-0011', '2024-05-01', '2025-05-01', 'Completo',  2950.00, 65000.00, 'ativa'),
    (32, 62, 2,  'APL-2024-0012', '2024-05-15', '2025-05-15', 'Terceiros',  890.00, 30000.00, 'ativa'),
], ['id', 'cliente_id', 'carro_id', 'numero_apolice', 'data_inicio', 'data_fim',
    'tipo_cobertura', 'valor_premio', 'valor_cobertura', 'status'])

novas_apolices.write.format('delta').mode('append').save(bronze_path('apolice'))

depois_ap = spark.read.format('delta').load(bronze_path('apolice')).count()
print(f'Apólices depois do INSERT: {depois_ap} (+{depois_ap - antes_ap})')

---
## 3. UPDATE — Reajuste de prêmios nas apólices

Aplicamos um reajuste de **10%** nas apólices com cobertura 'Completo' e status 'ativa'.

In [ ]:
dt_apolice = DeltaTable.forPath(spark, bronze_path('apolice'))

# Mostra alguns valores antes
print('=== Apólices Completo/ativa ANTES do UPDATE ===')
spark.read.format('delta').load(bronze_path('apolice')) \
    .filter((col('tipo_cobertura') == 'Completo') & (col('status') == 'ativa')) \
    .select('id', 'numero_apolice', 'tipo_cobertura', 'valor_premio', 'status') \
    .show(5)

# UPDATE via DeltaTable API (ACID)
dt_apolice.update(
    condition=(col('tipo_cobertura') == 'Completo') & (col('status') == 'ativa'),
    set={'valor_premio': col('valor_premio') * lit(1.10)}
)

print('=== Apólices Completo/ativa APÓS UPDATE (reajuste de 10%) ===')
spark.read.format('delta').load(bronze_path('apolice')) \
    .filter((col('tipo_cobertura') == 'Completo') & (col('status') == 'ativa')) \
    .select('id', 'numero_apolice', 'tipo_cobertura', 'valor_premio', 'status') \
    .show(5)

In [ ]:
# UPDATE via Spark SQL — marcar apólices vencidas como 'inativa'
spark.sql(f"""
    UPDATE delta.`{bronze_path('apolice')}`
    SET status = 'inativa'
    WHERE status = 'vencida'
""")

vencidas = spark.read.format('delta').load(bronze_path('apolice')) \
    .filter(col('status') == 'inativa').count()
print(f'Apólices marcadas como inativas: {vencidas}')

---
## 4. DELETE — Remover sinistros cancelados

Sinistros com status 'cancelado' são removidos da tabela Delta.

In [ ]:
dt_sinistro = DeltaTable.forPath(spark, bronze_path('sinistro'))

antes_s = spark.read.format('delta').load(bronze_path('sinistro')).count()
cancelados = spark.read.format('delta').load(bronze_path('sinistro')) \
    .filter(col('status') == 'cancelado').count()

print(f'Sinistros antes do DELETE: {antes_s}')
print(f'Sinistros com status cancelado: {cancelados}')

# DELETE via DeltaTable API
dt_sinistro.delete(condition=col('status') == 'cancelado')

depois_s = spark.read.format('delta').load(bronze_path('sinistro')).count()
print(f'Sinistros após DELETE: {depois_s} (-{antes_s - depois_s})')

In [ ]:
# DELETE via Spark SQL — remover sinistros de apólices inativas
spark.sql(f"""
    DELETE FROM delta.`{bronze_path('sinistro')}`
    WHERE apolice_id IN (
        SELECT id FROM delta.`{bronze_path('apolice')}`
        WHERE status = 'inativa'
    )
""")

final_s = spark.read.format('delta').load(bronze_path('sinistro')).count()
print(f'Sinistros após DELETE de apólices inativas: {final_s}')

---
## 5. DESCRIBE HISTORY — Histórico de transações

In [ ]:
print('=== Histórico de operações — tabela: cliente ===')
DeltaTable.forPath(spark, bronze_path('cliente')) \
    .history() \
    .select('version', 'timestamp', 'operation', 'operationMetrics') \
    .show(truncate=False)

In [ ]:
print('=== Histórico de operações — tabela: apolice ===')
DeltaTable.forPath(spark, bronze_path('apolice')) \
    .history() \
    .select('version', 'timestamp', 'operation', 'operationMetrics') \
    .show(truncate=False)

In [ ]:
print('=== Histórico de operações — tabela: sinistro ===')
DeltaTable.forPath(spark, bronze_path('sinistro')) \
    .history() \
    .select('version', 'timestamp', 'operation', 'operationMetrics') \
    .show(truncate=False)

---
## 6. Time Travel — Leitura em versão anterior

In [ ]:
print('=== Time Travel: tabela apolice na versão 0 (dados originais) ===')
df_v0 = spark.read.format('delta') \
    .option('versionAsOf', 0) \
    .load(bronze_path('apolice'))

print(f'Registros na versão 0: {df_v0.count()}')
df_v0.select('id', 'numero_apolice', 'valor_premio', 'status').show(5)

In [ ]:
# Comparação: versão 0 vs versão atual (diferença no valor_premio)
df_atual = spark.read.format('delta').load(bronze_path('apolice'))

print(f'Apólices na versão 0 (original): {df_v0.count()}')
print(f'Apólices na versão atual:         {df_atual.count()}')

avg_v0     = df_v0.agg({'valor_premio': 'avg'}).collect()[0][0]
avg_atual  = df_atual.agg({'valor_premio': 'avg'}).collect()[0][0]

print(f'\nPreço médio apólice v0:     R$ {avg_v0:.2f}')
print(f'Preço médio apólice atual:  R$ {avg_atual:.2f}')
print(f'Variação (reajuste 10%):    +{(avg_atual/avg_v0 - 1)*100:.1f}%')